# Serve Recommendation System

This notebook uses the trained model to recommend serve choices under specific match contexts.

The recommendation score combines predicted win probability, historical serve performance, and sample-size reliability.

In [ ]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("../data/processed/table_tennis_serves_features.csv")
model = joblib.load("../models/serve_win_probability_model.pkl")
model_features = joblib.load("../models/model_features.pkl")

In [ ]:
current_context = {"game_number":2,"server_score":8,"receiver_score":7,"game_state":"neutral","opponent_skill_level":"advanced","opponent_style":"looper","side":"backhand_side"}
current_context

This context represents the information known before the serve. The recommendation system will evaluate possible serve options under this situation.

In [ ]:
serve_options = df[["serve_type","spin_type","spin_intensity","serve_length","placement_zone","toss_height","contact_point","intended_setup"]].drop_duplicates()
recommendation_df = serve_options.copy()
for key, value in current_context.items():
    recommendation_df[key] = value

In [ ]:
# Recreate engineered features
recommendation_df["score_margin"] = recommendation_df["server_score"] - recommendation_df["receiver_score"]
recommendation_df["total_points_played_in_game"] = recommendation_df["server_score"] + recommendation_df["receiver_score"]
recommendation_df["is_tied"] = (recommendation_df["server_score"] == recommendation_df["receiver_score"]).astype(int)
recommendation_df["is_trailing"] = (recommendation_df["server_score"] < recommendation_df["receiver_score"]).astype(int)
recommendation_df["is_leading"] = (recommendation_df["server_score"] > recommendation_df["receiver_score"]).astype(int)
recommendation_df["is_late_game"] = (recommendation_df["total_points_played_in_game"] >= 16).astype(int)
recommendation_df["is_deuce_or_later"] = ((recommendation_df["server_score"] >= 10) & (recommendation_df["receiver_score"] >= 10)).astype(int)
recommendation_df["is_game_point_for_server"] = ((recommendation_df["server_score"] >= 10) & (recommendation_df["server_score"] > recommendation_df["receiver_score"])).astype(int)
recommendation_df["is_game_point_against_server"] = ((recommendation_df["receiver_score"] >= 10) & (recommendation_df["receiver_score"] > recommendation_df["server_score"])).astype(int)
recommendation_df["serve_spin_combo"] = recommendation_df["serve_type"] + "_" + recommendation_df["spin_type"]
recommendation_df["serve_length_spin_combo"] = recommendation_df["serve_length"] + "_" + recommendation_df["spin_type"]
recommendation_df["serve_placement_combo"] = recommendation_df["serve_type"] + "_" + recommendation_df["placement_zone"]
recommendation_df["full_serve_combo"] = recommendation_df["serve_type"] + "_" + recommendation_df["spin_type"] + "_" + recommendation_df["serve_length"] + "_" + recommendation_df["placement_zone"]
recommendation_df["is_heavy_spin"] = (recommendation_df["spin_intensity"] >= 3).astype(int)
recommendation_df["is_low_spin"] = (recommendation_df["spin_intensity"] <= 1).astype(int)

In [ ]:
combo_summary = df.groupby("full_serve_combo").agg(combo_attempts=("point_won", "count"), combo_win_rate=("point_won", "mean")).reset_index()
recommendation_df = recommendation_df.merge(combo_summary, on="full_serve_combo", how="left")
recommendation_df["combo_attempts"] = recommendation_df["combo_attempts"].fillna(0)
recommendation_df["combo_win_rate"] = recommendation_df["combo_win_rate"].fillna(df["point_won"].mean())
recommendation_df["combo_reliability"] = np.minimum(recommendation_df["combo_attempts"] / 30, 1)

In [ ]:
recommendation_df["predicted_win_probability"] = model.predict_proba(recommendation_df[model_features])[:, 1]

In [ ]:
recommendation_df["recommendation_score"] = 0.70 * recommendation_df["predicted_win_probability"] + 0.20 * recommendation_df["combo_win_rate"] + 0.10 * recommendation_df["combo_reliability"]

The recommendation score combines three pieces of information:

1. Model-predicted win probability
2. Historical win rate for that serve combination
3. Reliability based on sample size

This prevents the system from over-recommending serve combinations that performed well only once or twice.

In [ ]:
top_recommendations = recommendation_df.sort_values("recommendation_score", ascending=False)[["serve_type","spin_type","spin_intensity","serve_length","placement_zone","intended_setup","predicted_win_probability","combo_win_rate","combo_attempts","combo_reliability","recommendation_score"]].head(10)
top_recommendations

In [ ]:
def explain_recommendation(row):
    reasons = []
    if row["predicted_win_probability"] >= 0.60: reasons.append("high predicted win probability")
    if row["predicted_win_probability"] >= 0.55 and row["predicted_win_probability"] < 0.60: reasons.append("moderate predicted win probability")
    if row["combo_reliability"] >= 0.70: reasons.append("reliable historical sample")
    if row["combo_win_rate"] >= 0.60: reasons.append("strong historical win rate")
    if row["spin_intensity"] >= 3: reasons.append("uses heavy spin")
    if row["combo_attempts"] == 0: reasons.append("untested combination — use with caution")
    return ", ".join(reasons)

top_recommendations = top_recommendations.copy()
top_recommendations["reason"] = top_recommendations.apply(explain_recommendation, axis=1)
top_recommendations

## Score Distribution

We can look at how recommendation scores are distributed across all possible serve options. This helps us understand how discriminating the scoring formula is and where the top 10 threshold falls relative to the full distribution.

In [ ]:
import matplotlib.pyplot as plt

threshold_score = recommendation_df.sort_values("recommendation_score", ascending=False)["recommendation_score"].iloc[9]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(recommendation_df["recommendation_score"], bins=20, edgecolor="black")
ax.axvline(threshold_score, color="red", linestyle="--", label="Top 10 threshold")
ax.set_title("Distribution of Recommendation Scores")
ax.set_xlabel("Recommendation Score")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
contexts = [
    {"label":"Neutral point vs looper","game_number":1,"server_score":4,"receiver_score":4,"game_state":"neutral","opponent_skill_level":"advanced","opponent_style":"looper","side":"backhand_side"},
    {"label":"Late-game pressure vs attacker","game_number":3,"server_score":9,"receiver_score":9,"game_state":"pressure","opponent_skill_level":"advanced","opponent_style":"attacker","side":"forehand_side"},
    {"label":"Trailing vs chopper","game_number":2,"server_score":6,"receiver_score":9,"game_state":"trailing","opponent_skill_level":"advanced","opponent_style":"chopper","side":"backhand_side"}
]

context_results = {}

for ctx in contexts:
    label = ctx["label"]
    print(f"\n=== Context: {label} ===")

    rec = serve_options.copy()
    for key, value in ctx.items():
        if key != "label":
            rec[key] = value

    rec["score_margin"] = rec["server_score"] - rec["receiver_score"]
    rec["total_points_played_in_game"] = rec["server_score"] + rec["receiver_score"]
    rec["is_tied"] = (rec["server_score"] == rec["receiver_score"]).astype(int)
    rec["is_trailing"] = (rec["server_score"] < rec["receiver_score"]).astype(int)
    rec["is_leading"] = (rec["server_score"] > rec["receiver_score"]).astype(int)
    rec["is_late_game"] = (rec["total_points_played_in_game"] >= 16).astype(int)
    rec["is_deuce_or_later"] = ((rec["server_score"] >= 10) & (rec["receiver_score"] >= 10)).astype(int)
    rec["is_game_point_for_server"] = ((rec["server_score"] >= 10) & (rec["server_score"] > rec["receiver_score"])).astype(int)
    rec["is_game_point_against_server"] = ((rec["receiver_score"] >= 10) & (rec["receiver_score"] > rec["server_score"])).astype(int)
    rec["serve_spin_combo"] = rec["serve_type"] + "_" + rec["spin_type"]
    rec["serve_length_spin_combo"] = rec["serve_length"] + "_" + rec["spin_type"]
    rec["serve_placement_combo"] = rec["serve_type"] + "_" + rec["placement_zone"]
    rec["full_serve_combo"] = rec["serve_type"] + "_" + rec["spin_type"] + "_" + rec["serve_length"] + "_" + rec["placement_zone"]
    rec["is_heavy_spin"] = (rec["spin_intensity"] >= 3).astype(int)
    rec["is_low_spin"] = (rec["spin_intensity"] <= 1).astype(int)
    rec = rec.merge(combo_summary, on="full_serve_combo", how="left")
    rec["combo_attempts"] = rec["combo_attempts"].fillna(0)
    rec["combo_win_rate"] = rec["combo_win_rate"].fillna(df["point_won"].mean())
    rec["combo_reliability"] = np.minimum(rec["combo_attempts"] / 30, 1)
    rec["predicted_win_probability"] = model.predict_proba(rec[model_features])[:, 1]
    rec["recommendation_score"] = 0.70 * rec["predicted_win_probability"] + 0.20 * rec["combo_win_rate"] + 0.10 * rec["combo_reliability"]

    top5 = rec.sort_values("recommendation_score", ascending=False)[["serve_type","spin_type","spin_intensity","serve_length","placement_zone","recommendation_score"]].head(5)
    print(top5.to_string(index=False))

    context_results[label] = rec

## Comparing Serve Types Across Contexts

In [ ]:
pivot_rows = []
for label, rec in context_results.items():
    top_serve = rec.sort_values("recommendation_score", ascending=False).iloc[0]["serve_type"]
    mean_scores = rec.groupby("serve_type")["recommendation_score"].mean()
    row = mean_scores.rename(label)
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).T
pivot_df.index.name = "serve_type"
pivot_df.columns.name = "context"
pivot_df

## Summary

This notebook converts the predictive model into a serve recommendation system. For each match context, the system ranks possible serves using a 3-part scoring formula:

- **70%** model-predicted win probability (primary driver)
- **20%** historical win rate for that serve combination
- **10%** reliability score based on sample size (capped at 30 attempts)

Key takeaways:

- Recommendations differ meaningfully by context — the same serve that works well against a looper may not be optimal against an attacker or chopper.
- Untested combinations (zero historical attempts) should be tried cautiously, as their historical win rate is imputed from the overall average rather than real observations.
- As more match data is collected, the reliability of the recommendations should improve and imputed values will be replaced with real performance data.

The recommendation system should be treated as a decision-support tool rather than an automatic answer.